# Step 11 — Efficiency + Pareto Frontier (RQ4)

**Settings:** GPU T4, Internet ON. **No Kaggle dataset attachment needed** —
unlike the step10a/10b/10c grid notebooks, `build_model(cfg)` needs no
dataset at all (Step 11 measures parameters/FLOPs/latency/memory on
synthetic tensors, never trains). The only network dependency is the
torchvision ImageNet checkpoints (~45 MB ResNet-18, ~10 MB
MobileNetV3-Small) and `pip install -r requirements.txt`.

See `step_writeups/step11.txt` for the full status, the pre-registered
measurement axes (Section 0), and what must be transcribed from this
session's output afterwards.

## 1. GPU check + clone repo + install deps

In [ ]:
import torch, sys, os, subprocess
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('WARNING: no GPU detected (Settings > Accelerator > GPU T4 to fix) -- '
          'this session will still measure CPU latency (the edge proxy), but '
          'the GPU columns of efficiency_table.json will be absent from THIS '
          'session\'s environment block.')

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo ready at', os.getcwd())

## 2. Record hardware provenance

A separate cell so the write-up has the hardware info even if a later cell
fails partway through.

In [ ]:
import json as _json
import sys
sys.path.insert(0, os.getcwd())
from src.utils.efficiency import collect_env, gpu_clock_snapshot

ENV_ID = 'kaggle_t4' if torch.cuda.is_available() else 'kaggle_cpu_only'
cpu_env = collect_env(device='cpu')
print('=== CPU environment ===')
print(_json.dumps(cpu_env, indent=2, default=str))
if torch.cuda.is_available():
    gpu_env = collect_env(device='cuda')
    print('=== GPU environment ===')
    print(_json.dumps(gpu_env, indent=2, default=str))
    print('=== GPU clocks (best-effort) ===')
    print(_json.dumps(gpu_clock_snapshot(), indent=2, default=str))
print('ENV_ID for this session:', ENV_ID)

## 3. Generate grid configs + run the offline tests (no data needed)

In [ ]:
!python scripts/build_grid_configs.py
!python -m pytest -q tests/test_grid_configs.py tests/test_pareto.py tests/test_efficiency.py

## 4. Gate: parameter counts must match the committed grid

Rebuilds all 12 models from their reference configs and asserts trainable
params match `results/mvt_results.json`'s `n_params` for all 40 mvt cells —
free, strong cross-check against the numbers Step 10's training actually
recorded, in the spirit of `scripts/aggregate_grid.py`'s Step-6
reproducibility check. Must print `40/40 mvt cells match` before any timing
is trusted.

In [ ]:
!python scripts/efficiency_table.py --check-params-only

## 5. Measure — the only session-dependent numbers in this repo

Params/FLOPs are deterministic and get recomputed+diffed against any
existing file; latency/memory are measured under THIS session's hardware
and are exempt from the byte-identical-rerun invariant by design (see
`results/efficiency_table.json`'s `reproducibility` block once this
finishes). `--device both` measures GPU only if CUDA is actually available
(never silently substitutes CPU for a "GPU" number).
`--include-reference-backbones` adds the architecture-only ViT-B/16 /
DeiT-Tiny rows (never trained here, gated so a missing `timm` can't break
the run) that turn `docs/DEFENCE_BRIEF.md`'s cited parameter counts into an
in-repo measurement.

Uses `subprocess.run([sys.executable, ...])` rather than `!python ...` —
`step10a`'s Section 6 documents why: IPython leaves an unresolved
`{name}` in a `!` command as literal text instead of erroring, which this
avoids entirely.

In [ ]:
cmd = [sys.executable, 'scripts/efficiency_table.py',
       '--device', 'both', '--cpu-threads', '1,0',
       '--env-id', ENV_ID, '--include-reference-backbones',
       '--out', 'results/efficiency_table.json']
print('>>>', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 6. Stability re-run — the honest substitute for byte-identical

Latency/memory can never be byte-identical across runs by design (host
load, clocks, kernel selection all vary). This cell re-measures into a
SEPARATE file and prints the max relative per-key difference in `per_image`
median latency, so "session-dependent" becomes a measured number in
`step_writeups/step11.txt` Section 4.3/5 rather than an unquantified
disclaimer.

In [ ]:
cmd = [sys.executable, 'scripts/efficiency_table.py',
       '--device', 'both', '--cpu-threads', '1', '--warmup', '5', '--measure', '20',
       '--no-train-step', '--env-id', ENV_ID + '_rerun',
       '--out', 'results/efficiency_table_rerun.json']
subprocess.run(cmd, check=True)

import json as _json
a = _json.load(open('results/efficiency_table.json'))
b = _json.load(open('results/efficiency_table_rerun.json'))
max_diff = 0.0
max_diff_key = None
for key, units in b.get('measured', {}).items():
    per_image_b = units.get('per_image', {})
    per_image_a = a.get('measured', {}).get(key, {}).get('per_image', {})
    for profile, timing_b in per_image_b.items():
        timing_a = per_image_a.get(profile)
        if not timing_a:
            continue
        m_a = timing_a['latency_ms']['median']
        m_b = timing_b['latency_ms']['median']
        if m_a <= 0:
            continue
        diff_pct = 100.0 * abs(m_b - m_a) / m_a
        if diff_pct > max_diff:
            max_diff = diff_pct
            max_diff_key = f'{key}@{profile}'
print(f'max relative per_image median difference across the two runs: {max_diff:.2f}% ({max_diff_key})')
if max_diff > 10.0:
    print('WARNING: drift exceeds 10% -- note this explicitly in step_writeups/step11.txt Section 5.')

## 7. Preview the Pareto plots

No GPU needed for this cell; it reads the JSON `results/efficiency_table.json`
just wrote. The COMMITTED figures are regenerated locally after this
session's artifacts are merged (Section 8/9) — this is a preview only.

In [ ]:
!python scripts/pareto_plots.py --env {ENV_ID}
from IPython.display import Image, display
for p in ['results/pareto_params_vs_accuracy.png', 'results/pareto_latency_vs_auroc.png']:
    if os.path.exists(p):
        display(Image(filename=p))

## 8. Pack + push artifacts

In [ ]:
# Pack + push this session's artifacts (mirrors step10a Section 7 / step9-mini Section 9b).
import glob, hashlib, json as _json, zipfile

ARTIFACT_STEM = 'step11_efficiency'
ZIP = f'/kaggle/working/{ARTIFACT_STEM}_artifacts.zip'
ALL_FILES = [p for p in [
    'results/efficiency_table.json',
    'results/efficiency_table_rerun.json',
    'results/pareto_frontier.json',
    'results/pareto_params_vs_accuracy.png',
    'results/pareto_latency_vs_auroc.png',
] if os.path.exists(p)]
ALL_FILES += sorted(glob.glob('results/pareto_params_vs_accuracy__*.png'))
ALL_FILES += sorted(glob.glob('results/pareto_latency_vs_auroc__*.png'))
ALL_FILES += sorted(glob.glob('results/pareto_audit/*.png'))
ALL_FILES += (['results/pareto_audit/_manifest.json']
              if os.path.exists('results/pareto_audit/_manifest.json') else [])

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ALL_FILES:
        zf.write(p)
    manifest_lines = [
        f'{p}  {os.path.getsize(p)}B  sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
        for p in ALL_FILES
    ]
    zf.writestr('MANIFEST.txt', '\n'.join(manifest_lines))
print(f'wrote {ZIP} ({len(ALL_FILES)} files)')

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

if HAVE_SECRETS:
    ds_dir = f'/kaggle/working/{ARTIFACT_STEM}_dataset'
    os.makedirs(ds_dir, exist_ok=True)
    subprocess.run(['cp', ZIP, ds_dir], check=True)
    meta = {
        "title": f"{ARTIFACT_STEM}-artifacts",
        "id": f"{os.environ['KAGGLE_USERNAME']}/{ARTIFACT_STEM}-artifacts",
        "licenses": [{"name": "CC0-1.0"}],
    }
    _json.dump(meta, open(f'{ds_dir}/dataset-metadata.json', 'w'))
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir, '-m', 'update', '-q'])
    print(f'pushed to Kaggle dataset {ARTIFACT_STEM}-artifacts')
else:
    from IPython.display import FileLink, display
    display(FileLink(ZIP))
    print('no Kaggle Secrets found -- use the download link above instead.')

## After this session

1. Do **not** run here: `scripts/make_master_tables.py`, `scripts/make_results_master.py`,
   or treat this notebook's `pareto_plots.py` preview as the committed figures.
   Those run locally after this session's zip is merged into `results/`.
2. Locally: merge the zip into `results/efficiency_table.json` (if a local
   `local_cpu` block already exists from a prior dev run, this session's
   `--merge`-style write only replaces its OWN environment id — nothing is
   averaged across environments); then run, in order:
   `python scripts/pareto_plots.py`, `python scripts/make_master_tables.py`,
   `python scripts/make_results_master.py`.
3. Transcribe `step_writeups/step11.txt` Section 4 from the committed
   `results/efficiency_table.json` + `results/pareto_frontier.json` —
   **never from this notebook's stdout.**
4. Record Section 6's max relative drift and this session's environment
   block in `step_writeups/step11.txt` Section 4.3/5.
5. Tick `progress.txt`'s Step 11 boxes.